# NO-SEED MODEL — stats only, seeds excluded from features

In [ ]:
import ast
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier, XGBRegressor

warnings.simplefilter(action="ignore", category=UserWarning)

In [ ]:
final = (
    pd.read_csv('data_2026/final_features_W.csv')
    .rename(columns={
        'WTEAMID': 'W_TEAMID',
        'LTEAMID': 'L_TEAMID',
        'WSCORE': 'W_SCORE',
        'LSCORE': 'L_SCORE',
    })
)
# W_TEAMID is always the winner in historical tourney data
final['WIN_INDICATOR'] = 1

In [ ]:
# Columns with null values and their respective counts
{col: int(final[col].isna().sum()) for col in final.columns if final[col].isna().any()}

In [ ]:
drop_cols = ['W_CTWINS', 'W_AVERAGECTSCORE', 'L_CTWINS', 'L_AVERAGECTSCORE']
final = final.drop(columns=[c for c in drop_cols if c in final.columns])
final = final.drop(columns=[c for c in ['W_WLOCN','W_WLOCH','W_WLOCA','L_WLOCN','L_WLOCH','L_WLOCA'] if c in final.columns])  # variants

In [ ]:
parameters = {
    "n_estimators": [100, 200, 300, 400, 500],
    "learning_rate": [0.1, 0.2, 0.3, 0.4, 0.5],
    "max_depth": list(range(3, 6, 1)),
    "min_child_weight": list(range(1, 6, 1)),
}

In [ ]:
# W_SEED and L_SEED excluded so the model must rely entirely on season stats
NON_FEATURE_COLS = {'SEASON', 'WIN_INDICATOR', 'L_TEAMID', 'W_TEAMID', 'W_SCORE', 'L_SCORE',
                    'ROUND', 'L_REGION', 'W_REGION', 'W_SEED', 'L_SEED'}
feature_cols = [c for c in final.columns if c not in NON_FEATURE_COLS]

train = final[(final.SEASON >= 2010) & (final.SEASON <= 2023)].copy()
test = final[final.SEASON > 2023].copy()

BRACKET_SEASON = 2026  # Season to predict the bracket for

### Swap the W and L teams

In [ ]:
# Double training data with both W/L orientations — perfectly balanced, reproducible, 2x data
df = train.copy()
w_cols = sorted([c for c in df.columns if c.startswith('W_')])
l_cols = ['L_' + c[2:] for c in w_cols]

df_swapped = df.copy()
for w, l in zip(w_cols, l_cols):
    df_swapped[w] = df[l]
    df_swapped[l] = df[w]

train = pd.concat([df, df_swapped], ignore_index=True)
train['WIN_INDICATOR'] = (train['W_SCORE'] > train['L_SCORE']).astype(int)

In [ ]:
X_train = train[feature_cols]
y_train = train['WIN_INDICATOR']

all_rounds = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss'),
    param_grid=parameters,
    n_jobs=-1,
    scoring="accuracy",
    cv=5,
)
all_rounds.fit(X_train, y_train)

### Train spread and total score models

In [ ]:
# Spread = W_SCORE - L_SCORE (positive when W team wins, negative when L team wins)
# Total = W_SCORE + L_SCORE
y_train_spread = train['W_SCORE'] - train['L_SCORE']
y_train_total  = train['W_SCORE'] + train['L_SCORE']

spread_model_cv = GridSearchCV(
    estimator=XGBRegressor(eval_metric='rmse'),
    param_grid=parameters,
    n_jobs=-1,
    scoring='neg_mean_squared_error',
    cv=5,
)
spread_model_cv.fit(X_train, y_train_spread)
print(f'Spread model best params: {spread_model_cv.best_params_}')

total_model_cv = GridSearchCV(
    estimator=XGBRegressor(eval_metric='rmse'),
    param_grid=parameters,
    n_jobs=-1,
    scoring='neg_mean_squared_error',
    cv=5,
)
total_model_cv.fit(X_train, y_train_total)
print(f'Total model best params: {total_model_cv.best_params_}')

In [ ]:
def swap_wl(df):
    """Return a copy of df with all W_ and L_ columns swapped, preserving dtypes."""
    out = df.copy()
    w_cols = sorted([c for c in df.columns if c.startswith('W_')])
    l_cols = ['L_' + c[2:] for c in w_cols]
    for w, l in zip(w_cols, l_cols):
        out[w] = df[l]
        out[l] = df[w]
    return out

# Unbiased evaluation: each game tested in both orientations, WIN_INDICATOR derived from score
test_r1 = test[test.ROUND == 1].copy()
test_eval = pd.concat([test_r1, swap_wl(test_r1)], ignore_index=True)
test_eval['WIN_INDICATOR'] = (test_eval['W_SCORE'] > test_eval['L_SCORE']).astype(int)

test_eval['PRED_WIN_INDICATOR'] = all_rounds.predict(test_eval[feature_cols])

for season in sorted(test_eval.SEASON.unique()):
    acc = accuracy_score(
        test_eval[test_eval.SEASON == season]['WIN_INDICATOR'],
        test_eval[test_eval.SEASON == season]['PRED_WIN_INDICATOR'],
    )
    print(f'Accuracy {season}: {acc:.4f}')
    globals()[f'accuracy_{season}'] = acc

accuracy = accuracy_score(test_eval['WIN_INDICATOR'], test_eval['PRED_WIN_INDICATOR'])
print(f'Accuracy total: {accuracy:.4f}')

#### Save the trained model

In [ ]:
optimal_spread_model = spread_model_cv.best_estimator_
optimal_total_model  = total_model_cv.best_estimator_

joblib.dump(optimal_spread_model, 'model/spread_model_no_seed_W.joblib')
joblib.dump(optimal_total_model,  'model/total_model_no_seed_W.joblib')
print('Spread and total (no-seed) models saved.')

# Evaluate on test set
test_eval['PRED_SPREAD'] = optimal_spread_model.predict(test_eval[feature_cols])
test_eval['PRED_TOTAL']  = optimal_total_model.predict(test_eval[feature_cols])
test_eval['ACTUAL_SPREAD'] = test_eval['W_SCORE'] - test_eval['L_SCORE']
test_eval['ACTUAL_TOTAL']  = test_eval['W_SCORE'] + test_eval['L_SCORE']

for season in sorted(test_eval.SEASON.unique()):
    sub = test_eval[test_eval.SEASON == season]
    print(f"{season}  spread MAE: {mean_absolute_error(sub.ACTUAL_SPREAD, sub.PRED_SPREAD):.2f} pts  "
          f"total MAE: {mean_absolute_error(sub.ACTUAL_TOTAL, sub.PRED_TOTAL):.2f} pts")

print(f"Overall spread MAE: {mean_absolute_error(test_eval.ACTUAL_SPREAD, test_eval.PRED_SPREAD):.2f} pts")
print(f"Overall total  MAE: {mean_absolute_error(test_eval.ACTUAL_TOTAL,  test_eval.PRED_TOTAL):.2f} pts")

In [ ]:
import os
os.makedirs('model', exist_ok=True)

optimal_model = all_rounds.best_estimator_
joblib.dump(optimal_model, 'model/march_madness_model_no_seed_W.joblib')
print(f'Win model (no-seed) saved. Best params: {all_rounds.best_params_}')
print(f'Test seasons: {sorted(test_eval.SEASON.unique())}')

import os
os.makedirs('model', exist_ok=True)

optimal_model = all_rounds.best_estimator_
joblib.dump(optimal_model, 'model/march_madness_model_W.joblib')
print(f'Model saved. Best params: {all_rounds.best_params_}')
print(f'Test seasons: {sorted(test_eval.SEASON.unique())}')

In [ ]:
def predict_games(win_model, spread_model, total_model, df, feature_cols):
    """Run all three model predictions, filling missing feature cols with 0."""
    df = df.copy()
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0
    df['PRED_WIN_INDICATOR'] = win_model.predict(df[feature_cols])
    df['PRED_SPREAD'] = spread_model.predict(df[feature_cols]).round(1)
    df['PRED_TOTAL']  = total_model.predict(df[feature_cols]).round(1)
    return df


def add_team_names(result, teams):
    """Join team names onto a result dataframe that has W_TEAMID and L_TEAMID."""
    result = result.merge(
        teams[['TEAMID', 'TEAMNAME']], left_on='L_TEAMID', right_on='TEAMID'
    ).rename(columns={'TEAMNAME': 'L_TEAM_NAME'}).drop(columns=['TEAMID'])
    result = result.merge(
        teams[['TEAMID', 'TEAMNAME']], left_on='W_TEAMID', right_on='TEAMID'
    ).rename(columns={'TEAMNAME': 'W_TEAM_NAME'}).drop(columns=['TEAMID'])
    return result


def get_round_winners(result):
    """Pick the predicted winner from each game, returning (W_TEAM_ID, W_TEAM_NAME, W_SEED, W_REGION)."""
    r = result.copy()
    r['W_TEAM_ID']   = np.where(r.PRED_WIN_INDICATOR == 1, r.W_TEAMID,    r.L_TEAMID)
    r['W_TEAM_NAME'] = np.where(r.PRED_WIN_INDICATOR == 1, r.W_TEAM_NAME, r.L_TEAM_NAME)
    r['W_SEED']      = np.where(r.PRED_WIN_INDICATOR == 1, r.W_SEED,      r.L_SEED)
    r['W_REGION']    = np.where(r.PRED_WIN_INDICATOR == 1, r.W_REGION,    r.L_REGION)
    return r[['W_TEAM_ID', 'W_TEAM_NAME', 'W_SEED', 'W_REGION']]


def add_season_stats(matchups, season_stats, season_year):
    """Join season stats onto matchups that have WTEAMID2 and LTEAMID2 columns."""
    season = season_stats[season_stats.SEASON == season_year].drop(columns=['SEASON'])
    season = season.drop(columns=[c for c in ['REGION'] if c in season.columns])
    season_w = season.copy()
    season_w.columns = ['W_' + c for c in season_w.columns]
    season_l = season.copy()
    season_l.columns = ['L_' + c for c in season_l.columns]

    df = matchups.merge(season_w, left_on='WTEAMID2', right_on='W_TEAMID').drop(columns=['W_TEAMID'])
    df = df.merge(season_l, left_on='LTEAMID2', right_on='L_TEAMID').drop(columns=['L_TEAMID'])
    df = df.rename(columns={'WTEAMID2': 'W_TEAMID', 'LTEAMID2': 'L_TEAMID'})
    return df


teams = pd.read_csv('data_2026/WTeams.csv')
teams.columns = teams.columns.str.upper()

season_stats = pd.read_csv('data_2026/final_season_stats_W.csv')

In [ ]:
# Load seeds for the bracket season
seeds_raw = pd.read_csv('data_2026/WNCAATourneySeeds.csv')
seeds_raw.columns = seeds_raw.columns.str.upper()
seeds_bracket = seeds_raw[seeds_raw.SEASON == BRACKET_SEASON].copy()
seeds_bracket['REGION'] = seeds_bracket['SEED'].str[0]
seeds_bracket['IS_PLAYIN'] = seeds_bracket['SEED'].str[1:].str.contains('[ab]', regex=True)
seeds_bracket['SEED_NUM'] = seeds_bracket['SEED'].str[1:].str.replace('[a-z]', '', regex=True).astype(int)

playin_seeds = seeds_bracket[seeds_bracket.IS_PLAYIN][['TEAMID', 'REGION', 'SEED_NUM']].rename(columns={'SEED_NUM': 'SEED'})
regular_seeds = seeds_bracket[~seeds_bracket.IS_PLAYIN][['TEAMID', 'REGION', 'SEED_NUM']].rename(columns={'SEED_NUM': 'SEED'})

playin_rows = []
for (region, seed), group in playin_seeds.groupby(['REGION', 'SEED']):
    team_ids = group.TEAMID.tolist()
    if len(team_ids) == 2:
        playin_rows.append({
            'WTEAMID2': team_ids[0], 'LTEAMID2': team_ids[1],
            'W_SEED': seed, 'L_SEED': seed,
            'W_REGION': region, 'L_REGION': region,
        })

if playin_rows:
    playin_matchups = pd.DataFrame(playin_rows)
    games_playin = add_season_stats(playin_matchups, season_stats, BRACKET_SEASON)
    result_playin = predict_games(optimal_model, optimal_spread_model, optimal_total_model, games_playin, feature_cols)
    result_playin = add_team_names(result_playin, teams)
    playin_result = result_playin[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
    playin_winners = get_round_winners(playin_result)
    print(f'Play-in results ({BRACKET_SEASON}):')
    display(playin_result[['W_TEAM_NAME', 'L_TEAM_NAME', 'W_SEED', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']].sort_values('W_REGION'))
else:
    playin_winners = pd.DataFrame(columns=['W_TEAM_ID', 'W_TEAM_NAME', 'W_SEED', 'W_REGION'])
    print(f'No play-in games found for {BRACKET_SEASON}')

## Round of 64

In [ ]:
ROUND_1_PAIRS = [(1, 16), (8, 9), (5, 12), (4, 13), (6, 11), (3, 14), (7, 10), (2, 15)]

round1_seeds = regular_seeds.copy()
if not playin_winners.empty:
    playin_w_seeds = playin_winners.rename(
        columns={'W_TEAM_ID': 'TEAMID', 'W_SEED': 'SEED', 'W_REGION': 'REGION'}
    )[['TEAMID', 'REGION', 'SEED']]
    round1_seeds = pd.concat([round1_seeds, playin_w_seeds], ignore_index=True)

round1_rows = []
for region in round1_seeds.REGION.unique():
    r = round1_seeds[round1_seeds.REGION == region].set_index('SEED')
    for s1, s2 in ROUND_1_PAIRS:
        if s1 in r.index and s2 in r.index:
            round1_rows.append({
                'WTEAMID2': r.loc[s1, 'TEAMID'], 'LTEAMID2': r.loc[s2, 'TEAMID'],
                'W_SEED': s1, 'L_SEED': s2,
                'W_REGION': region, 'L_REGION': region,
            })

round1_matchups = pd.DataFrame(round1_rows)
games_r1 = add_season_stats(round1_matchups, season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, optimal_spread_model, optimal_total_model, games_r1, feature_cols)
result = add_team_names(result, teams)
print(f'Round of 64 — {BRACKET_SEASON}: {len(result)} games')

In [ ]:
round_1_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
round_1_results.sort_values('W_REGION')

# Winners round 1

In [ ]:
round_1_winners = get_round_winners(round_1_results)
round_1_winners.sort_values('W_REGION')

In [ ]:
ROUND_2_SEED_PAIRS = [
    (1,8),(1,9),(16,8),(16,9),
    (4,5),(4,12),(13,5),(13,12),
    (3,6),(3,11),(14,6),(14,11),
    (2,7),(2,10),(15,7),(15,10),
]

df1 = round_1_winners.rename(columns={'W_TEAM_NAME':'W_TEAM_NAME','W_REGION':'W_REGION','W_TEAM_ID':'WTEAMID2','W_SEED':'W_SEED'})
df2 = round_1_winners.rename(columns={'W_TEAM_NAME':'L_TEAM_NAME','W_REGION':'L_REGION','W_TEAM_ID':'LTEAMID2','W_SEED':'L_SEED'})

cross = df1.merge(df2, how='cross', suffixes=('','_r'))
seed_set = set(ROUND_2_SEED_PAIRS)
mask = (
    (cross.W_REGION == cross.L_REGION) &
    cross.apply(lambda r: (r.W_SEED, r.L_SEED) in seed_set, axis=1)
)
second_round_matchups = cross[mask][['W_TEAM_NAME','W_REGION','WTEAMID2','W_SEED','L_TEAM_NAME','L_REGION','LTEAMID2','L_SEED']]
second_round_matchups.sort_values('W_REGION')

# ROUND OF 32!!!

In [ ]:
second_round_matchups[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
).sort_values('REGION')

In [ ]:
games_32 = add_season_stats(second_round_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, optimal_spread_model, optimal_total_model, games_32, feature_cols)
result = add_team_names(result, teams)

round_2_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
round_2_results

In [ ]:
round_2_winners = get_round_winners(round_2_results)
round_2_winners.sort_values('W_REGION')

# Round 2 winners

In [ ]:
ROUND_3_SEED_PAIRS = [
    (s1, s2)
    for s1 in [1,16,8,9]
    for s2 in [5,12,4,13]
] + [
    (s1, s2)
    for s1 in [6,11,3,14]
    for s2 in [7,10,2,15]
]

df1 = round_2_winners.rename(columns={'W_TEAM_NAME':'W_TEAM_NAME','W_REGION':'W_REGION','W_TEAM_ID':'WTEAMID2','W_SEED':'W_SEED'})
df2 = round_2_winners.rename(columns={'W_TEAM_NAME':'L_TEAM_NAME','W_REGION':'L_REGION','W_TEAM_ID':'LTEAMID2','W_SEED':'L_SEED'})

cross = df1.merge(df2, how='cross', suffixes=('','_r'))
seed_set = set(ROUND_3_SEED_PAIRS)
mask = (
    (cross.W_REGION == cross.L_REGION) &
    cross.apply(lambda r: (r.W_SEED, r.L_SEED) in seed_set, axis=1)
)
third_round_matchups = cross[mask][['W_TEAM_NAME','W_REGION','WTEAMID2','W_SEED','L_TEAM_NAME','L_REGION','LTEAMID2','L_SEED']]
third_round_matchups.sort_values('W_REGION')

# SWEET SIXTEEN!!!

In [ ]:
third_round_matchups[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
).sort_values('REGION')

In [ ]:
games_16 = add_season_stats(third_round_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, optimal_spread_model, optimal_total_model, games_16, feature_cols)
result = add_team_names(result, teams)

sweet_16_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
sweet_16_winners = get_round_winners(sweet_16_results)
sweet_16_results

# Sweet 16 winners!

In [ ]:
ELITE_8_SEED_PAIRS = [
    (s1, s2)
    for s1 in [1,16,8,9,5,12,4,13]
    for s2 in [6,11,3,14,7,10,2,15]
]

df1 = sweet_16_winners.rename(columns={'W_TEAM_NAME':'W_TEAM_NAME','W_REGION':'W_REGION','W_TEAM_ID':'WTEAMID2','W_SEED':'W_SEED'})
df2 = sweet_16_winners.rename(columns={'W_TEAM_NAME':'L_TEAM_NAME','W_REGION':'L_REGION','W_TEAM_ID':'LTEAMID2','W_SEED':'L_SEED'})

cross = df1.merge(df2, how='cross', suffixes=('','_r'))
seed_set = set(ELITE_8_SEED_PAIRS)
mask = (
    (cross.W_REGION == cross.L_REGION) &
    cross.apply(lambda r: (r.W_SEED, r.L_SEED) in seed_set, axis=1)
)
elite_8_matchups = cross[mask][['W_TEAM_NAME','W_REGION','WTEAMID2','W_SEED','L_TEAM_NAME','L_REGION','LTEAMID2','L_SEED']]
elite_8_matchups.sort_values('W_REGION')

# Elite 8

In [ ]:
elite_8_matchups[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
).sort_values('REGION')

In [ ]:
elite_8 = add_season_stats(elite_8_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, optimal_spread_model, optimal_total_model, elite_8, feature_cols)
result = add_team_names(result, teams)

elite_8_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
elite_8_winners = get_round_winners(elite_8_results)
elite_8_results

# Elite 8 winners

In [ ]:
# Final four matchups: W vs X regions, Y vs Z regions
wx = elite_8_winners[elite_8_winners.W_REGION == 'W'].merge(
    elite_8_winners[elite_8_winners.W_REGION == 'X'].rename(
        columns={'W_TEAM_ID':'TEAM_2','W_TEAM_NAME':'W_TEAM_NAME_2','W_SEED':'SEED_2','W_REGION':'REGION_2'}
    ),
    how='cross',
)

yz = elite_8_winners[elite_8_winners.W_REGION == 'Y'].merge(
    elite_8_winners[elite_8_winners.W_REGION == 'Z'].rename(
        columns={'W_TEAM_ID':'TEAM_2','W_TEAM_NAME':'W_TEAM_NAME_2','W_SEED':'SEED_2','W_REGION':'REGION_2'}
    ),
    how='cross',
)

final_four_matchups = pd.concat([wx, yz]).rename(columns={
    'W_TEAM_ID': 'WTEAMID2',
    'W_TEAM_NAME_2': 'L_TEAM_NAME',
    'REGION_2': 'L_REGION',
    'TEAM_2': 'LTEAMID2',
    'SEED_2': 'L_SEED',
})[['W_TEAM_NAME','W_REGION','WTEAMID2','W_SEED','L_TEAM_NAME','L_REGION','LTEAMID2','L_SEED']]

# FINAL FOUR!

In [ ]:
final_four_matchups[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
).sort_values('REGION')

In [ ]:
final_four = add_season_stats(final_four_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, optimal_spread_model, optimal_total_model, final_four, feature_cols)
result = add_team_names(result, teams)

final_four_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
final_four_winners = get_round_winners(final_four_results)
final_four_results

In [ ]:
# Championship: cross-join final four winners and keep the one unique matchup
championship_matchup = (
    final_four_winners.rename(columns={'W_TEAM_ID':'WTEAMID2','W_TEAM_NAME':'W_TEAM_NAME','W_SEED':'W_SEED','W_REGION':'W_REGION'})
    .merge(
        final_four_winners.rename(columns={'W_TEAM_ID':'LTEAMID2','W_TEAM_NAME':'L_TEAM_NAME','W_SEED':'L_SEED','W_REGION':'L_REGION'}),
        how='cross',
    )
    .query('WTEAMID2 != LTEAMID2')
    .head(1)
)

# CHAMPIONSHIP!

In [ ]:
championship_matchup[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
)

In [ ]:
championship = add_season_stats(championship_matchup[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, optimal_spread_model, optimal_total_model, championship, feature_cols)
result = add_team_names(result, teams)

championship_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
championship_results

In [ ]:
champion = get_round_winners(championship_results)
champion

# CHAMPION!

## Getting started with 2026 predictions

Before running the bracket prediction cells, you need 2026 tournament seed data in `data_2026/MNCAATourneySeeds.csv`.

Once the seeds are added, re-run `1_cleaning_2026.ipynb` to regenerate `final_season_stats.csv` with 2026 season data, then run this notebook top to bottom.